# DeepGuard — WildDeepfake full download (WebDataset-safe)

The Hugging Face mirror stores WildDeepfake as WebDataset shards. Some files have a `.tar.gz` name but are not gzip-compressed. This version therefore auto-detects the tar format (`r:*`) instead of forcing gzip, validates each shard, and extracts it safely.


In [1]:
from google.colab import drive
from pathlib import Path
import shutil, json
drive.mount('/content/drive', force_remount=False)
ROOT=Path('/content/drive/MyDrive/DeepGuard')
BASE=ROOT/'datasets/WildDeepfake/full'
ARCH=BASE/'archives'; OUT=BASE/'extracted'
ARCH.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
print('Destination:',BASE)
print('Free Drive GiB:',round(shutil.disk_usage('/content/drive').free/1024**3,1))
assert shutil.disk_usage('/content/drive').free > 15*1024**3, 'Less than 15 GB free on Drive'


Mounted at /content/drive
Destination: /content/drive/MyDrive/DeepGuard/datasets/WildDeepfake/full
Free Drive GiB: 83.2


In [2]:
!pip -q install -U huggingface_hub
from huggingface_hub import list_repo_files, hf_hub_download
repo='xingjunm/WildDeepfake'
all_files=list_repo_files(repo_id=repo,repo_type='dataset')
targets=[f for f in all_files if f.startswith('deepfake_in_the_wild/') and f.endswith('.tar.gz')]
print('Archives found:',len(targets))
assert len(targets)>100, 'Unexpected repository listing; stopping.'
(BASE/'remote_files.json').write_text(json.dumps(targets,indent=2))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 24.0 MB/s eta 0:00:00
Archives found: 1120


53278

In [ ]:
# Download + validate each WebDataset shard.
import tarfile
manifest_path=BASE/'download_manifest.json'
manifest={}
for i,remote in enumerate(targets,1):
    category=Path(remote).parent.name
    original=Path(remote).name
    target=ARCH/f'{category}__{original}'
    valid=False
    if target.exists() and target.stat().st_size>0:
        try:
            with tarfile.open(target,'r:*') as tf:
                first=next(iter(tf),None)
            valid=first is not None
        except Exception:
            print('Removing invalid previous file:',target.name)
            target.unlink(missing_ok=True)
    if not valid:
        print(f'[{i}/{len(targets)}] downloading {remote}')
        cached=hf_hub_download(repo_id=repo,filename=remote,repo_type='dataset',local_dir=str(BASE/'hf_cache'))
        p=Path(cached)
        shutil.copy2(p,target)
        try:
            with tarfile.open(target,'r:*') as tf:
                first=next(iter(tf),None)
            if first is None: raise RuntimeError('empty archive')
        except Exception as e:
            target.unlink(missing_ok=True)
            raise RuntimeError(f'Downloaded file is not a readable tar/WebDataset shard: {remote}: {e}')
    manifest[remote]={'local':str(target),'bytes':target.stat().st_size}
    manifest_path.write_text(json.dumps(manifest,indent=2))
print('VALID SHARDS:',len(manifest))


In [ ]:
# Extract validated shards; compression is auto-detected.
import tarfile
for i,p in enumerate(sorted(ARCH.glob('*.tar.gz')),1):
    dest=OUT/p.name[:-7]
    marker=dest/'.EXTRACTED_OK'
    if marker.exists():
        print(f'[{i}] already extracted {p.name}')
        continue
    dest.mkdir(parents=True,exist_ok=True)
    print(f'[{i}] extracting {p.name}')
    try:
        with tarfile.open(p,'r:*') as tf: tf.extractall(dest, filter='data')
        marker.write_text('ok')
    except Exception as e:
        print('FAILED shard:',p,'error:',repr(e))
        raise
print('Extraction complete.')


In [ ]:
from collections import Counter
counts=Counter(); total=0
for p in OUT.rglob('*'):
    if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png'}:
        total+=p.stat().st_size; s=str(p).lower()
        counts['fake' if 'fake_test' in s else 'real' if 'real_test' in s else 'other']+=1
print('Images:',sum(counts.values()))
print('Counts:',dict(counts))
print('Extracted GiB:',round(total/1024**3,2))
print('DONE — WildDeepfake is ready for DeepGuard testing.')
